![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Activity: Train YOLO on Your Own Dataset

In Lab 1 you ran a YOLO that already knew 80 everyday objects. Now you'll teach
YOLO to detect **your own** objects: collect and label images in **Roboflow**,
then **fine-tune** YOLO on them here in Colab.

## Part 1 -- Build your dataset in Roboflow (in your browser)

1. Go to **[roboflow.com](https://roboflow.com)** and create a free account.
2. Click **Create New Project** and set the type to **Object Detection**.
3. **Upload** your images -- about 20-50 photos of the object(s) you want to detect.
4. For each image, **draw a box** around every object and give it a **class name**
   (for example `cat`, `cup`, `phone`).
5. Click **Generate** to make a dataset version (Roboflow auto-splits it into
   train / valid / test).
6. Click **Export Dataset**, choose the **YOLOv8** format, and pick
   **"Download zip to your computer"**.

You now have a `.zip` file -- upload it in Part 2.

## Part 2 -- Train in Colab

### Setup

*Use a GPU runtime: Runtime -> Change runtime type -> GPU.*

In [ ]:
%pip install ultralytics -q

### 1) Upload your Roboflow dataset

Run the cell, then choose the **.zip** you exported from Roboflow.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zipfile.ZipFile(next(iter(uploaded))).extractall("dataset")

### 2) Point YOLO at your data

This finds the dataset's `data.yaml` and fixes its folder paths.

In [ ]:
import yaml, glob, os

yaml_path = glob.glob("dataset/**/data.yaml", recursive=True)[0]
root = os.path.dirname(os.path.abspath(yaml_path))
cfg = yaml.safe_load(open(yaml_path))
cfg["path"] = root
cfg["train"] = "train/images"
cfg["val"] = "valid/images" if os.path.isdir(f"{root}/valid/images") else "val/images"
yaml.safe_dump(cfg, open(yaml_path, "w"))
print("classes:", cfg["names"])

### 3) Fine-tune YOLO on your objects

We start from a YOLO already trained on everyday objects and adapt it to yours.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")               # pretrained starting point
model.train(data=yaml_path, epochs=50, imgsz=640)

### 4) A helper to show predictions

In [ ]:
import matplotlib.pyplot as plt

def show(results):
    plt.figure(figsize=(8, 8))
    plt.imshow(results[0].plot()[:, :, ::-1])    # draw boxes, BGR -> RGB
    plt.axis("off"); plt.show()

### 5) Test it on a new image

Run the cell and upload a photo that contains your object(s).

In [ ]:
from google.colab import files

uploaded = files.upload()
test_image = next(iter(uploaded))
show(model(test_image))

### *Contributed By: Sattam Altwaim*